In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import uuid
from fake_useragent import UserAgent
import numpy as np
from io import StringIO
import re


c:\Users\MS1\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\__init__.py


In [2]:
global url_general
global df_Competition, df_Match, df_Team_game,df_Team

df_Competition=pd.DataFrame(columns=['id_competition', 'name_competition', 'season','level','type','category'])
df_Fase = pd.DataFrame(columns=['id_fase', 'id_competition', 'phase_name'])

df_Match=pd.DataFrame(columns=['id_match','id_match_web', 'id_fase', 'round','url_video','start_date','start_time','id_stadium','attendance','name_match','result'])
df_Team_game=pd.DataFrame(columns=['id_match', 'id_team','id_team_match', 'name_team','result','goals','goals_conceded'])
df_Team=pd.DataFrame(columns=['id_team','id_team_web', 'name_team','nickname','founded','home_kit', 'goalkeeper_kit','away_kit','alternate_colours','phone','email','url_logo','url_web','club_sponsor'])
df_Stadium=pd.DataFrame(columns=['id_stadium', 'stadium_name','capacity','covered_seats','stadium_adress','url_map', 'type_of_pitch','dimensions'])
df_Referee=pd.DataFrame(columns=['id_referee', 'name_referee'])
df_Referee_game=pd.DataFrame(columns=['id_match', 'id_referee','type'])
df_Player_game=pd.DataFrame(columns=[])
df_Player=pd.DataFrame(columns=['id_player', 'name_player','position','age','place_of_birth','joined'])
df_Staff_team=pd.DataFrame(columns=['role', 'id_Staff','id_team'])
df_Staff=pd.DataFrame(columns=['id_Staff', 'name_staff'])

df_Action_document=pd.DataFrame(columns=['id_action_document','id_player_game','time','half','type'])


url_general= 'https://southern-football-league.co.uk/'
ua = UserAgent()


In [3]:
def Referee(id_match,name_referee):
    global df_Referee,df_Referee_game
    type=''#Tipo de arbitro: si es principal, linier...
    if not name_referee:
        id_referee=''
        
    elif (not df_Referee['name_referee'].isin([name_referee]).any()):
        id_referee=uuid.uuid4()

        df_Referee_aux=pd.DataFrame([[id_referee,name_referee]],columns=['id_referee', 'name_referee'])
        df_Referee = pd.concat([df_Referee, df_Referee_aux], ignore_index=True)
        df_Referee.to_excel(r'..\Maestros_BD\Referee.xlsx',index=False,sheet_name='Referee')

    else:
        id_referee=df_Referee[(df_Referee['name_referee']==name_referee)]['id_referee'].values[0]

    df_Referee_game_aux=pd.DataFrame([[id_match,id_referee,type]],columns=['id_match', 'id_referee','type'])
    df_Referee_game = pd.concat([df_Referee_game, df_Referee_game_aux], ignore_index=True)
    df_Referee_game.to_excel(r'..\Maestros_BD\Referee_game.xlsx',index=False,sheet_name='Referee_game')




In [4]:
import uuid

def Competition(bs_inicio, competition):
    global df_Competition, df_Fase

    # Obtener temporada
    select_tag = bs_inicio.find('select', id='choice2')
    selected_option = select_tag.find('option', selected=True)
    season = selected_option.text.strip()

    # 1. Intentar coincidencia exacta en la competición
    df_temp = df_Competition[df_Competition['season'] == season]
    exact_match = df_temp[df_temp['name_competition'] == competition]

    if not exact_match.empty:
        # Coincidencia exacta -> usar fase "Main"
        id_competition = exact_match.iloc[0]['id_competition']
        phase_name = "Main"

    else:
        # 2. Buscar coincidencia parcial (competición base + fase)
        base_name = None
        for _, row in df_temp.iterrows():
            if competition.startswith(row['name_competition']):
                base_name = row['name_competition']
                id_competition = row['id_competition']
                phase_name = competition.replace(base_name, '').strip()
                break

        if base_name is None:
            # 3. No se encontró ninguna competición existente -> crearla
            id_competition = str(uuid.uuid4())  # UUID para id_competition
            base_name = competition  # Usamos el parámetro competition como nombre base
            phase_name = "Main"
            
            # Pedir datos por input
            print("Nueva competición detectada. Por favor introduce los datos:")
            name_competition_input = input(f'Introduce el nombre de la competicion en la web es {competition}').strip()
            level_input = input("Introduce el nivel de la competicion (Solo competiciones de liga):").strip()
            type_input = input("Introduce el tipo de competicion (League/FA/League cup/Community Shield...): ").strip()
            category_input = input("Introduce la categoria (Senior,fem,u18...):").strip()

            print(f'Competicion: {name_competition_input}')
            print(f'Nivel: {level_input}')
            print(f'Tipo: {type_input}')
            print(f'Categoria: {category_input}')

            new_comp = {
                'id_competition': id_competition,
                'name_competition': name_competition_input or base_name,
                'season': season,
                'level': level_input if level_input else None,
                'type': type_input if type_input else None,
                'category': category_input if category_input else None
            }
            df_Competition = pd.concat([df_Competition, pd.DataFrame([new_comp])], ignore_index=True)
            df_Competition.to_excel(r'..\Maestros_BD\Competition.xlsx', index=False, sheet_name='Competition')

    # Buscar fase correspondiente
    fase_match = df_Fase[
        (df_Fase['id_competition'] == id_competition) &
        (df_Fase['phase_name'] == phase_name)
    ]

    if not fase_match.empty:
        id_fase = fase_match.iloc[0]['id_fase']
    else:
        # Crear nueva fase
        id_fase = str(uuid.uuid4())  # UUID para id_fase
        new_fase = {
            'id_fase': id_fase,
            'id_competition': id_competition,
            'phase_name': phase_name
        }
        df_Fase = pd.concat([df_Fase, pd.DataFrame([new_fase])], ignore_index=True)
        df_Fase.to_excel(r'..\Maestros_BD\Fase.xlsx', index=False, sheet_name='Fase')

    return id_fase







In [5]:
def Action_document(df_local, df_visitante):
    global df_Action_document
    rows = []
    df = pd.concat([df_local, df_visitante], ignore_index=True)
    
    sufijo_tipo_map = {
        'p': 'P',
        'o': 'OG',
        # otros sufijos si tienes
    }

    for idx, row in df.iterrows():
        player_name = row['Player']
        id_player_game = row['id_player_game']

        minutos_str = row.get('GM', '')

        if pd.notna(minutos_str) and minutos_str != '':
            # Procesamos minutos con sufijo etc. (igual que antes)
            minutos_raw = [m.strip() for m in str(minutos_str).split(',')]

            minutos = []
            minutos_con_tipo = {}
            minutos_sin_tipo = []

            for m in minutos_raw:
                match = re.match(r"(\d+)([a-zA-Z]*)'?$", m)
                if match:
                    numero = match.group(1)
                    sufijo = match.group(2).lower()
                    minutos.append(numero)
                    if sufijo in sufijo_tipo_map:
                        minutos_con_tipo[numero] = sufijo_tipo_map[sufijo]
                    else:
                        minutos_sin_tipo.append(numero)
                else:
                    minutos.append(m)
                    minutos_sin_tipo.append(m)

            conteo_tipos = {}
            for tipo in ['G', 'P', 'OG']:
                valor = row.get(tipo)
                if pd.notna(valor) and str(valor).strip() != '':
                    partes = str(valor).replace(',', ' ').split()
                    conteo_tipos[tipo] = len(partes)
                else:
                    conteo_tipos[tipo] = 0

            asignados_por_tipo = {tipo: 0 for tipo in ['G', 'P', 'OG']}
            for tipo_asignado in minutos_con_tipo.values():
                asignados_por_tipo[tipo_asignado] += 1

            tipo_minutos_restantes = []
            for tipo in ['G', 'P', 'OG']:
                restantes = conteo_tipos.get(tipo, 0) - asignados_por_tipo.get(tipo, 0)
                if restantes > 0:
                    tipo_minutos_restantes.extend([tipo] * restantes)

            if len(minutos_sin_tipo) != len(tipo_minutos_restantes):
                if len(set(tipo_minutos_restantes)) <= 1:
                    tipo_minutos_restantes = tipo_minutos_restantes * len(minutos_sin_tipo)
                else:
                    print(f"⚠️ Fila {idx} ({player_name}): asignación aleatoria de tipos a minutos sin sufijo")
                    while len(tipo_minutos_restantes) < len(minutos_sin_tipo):
                        tipo_minutos_restantes.append(random.choice(tipo_minutos_restantes))
                    tipo_minutos_restantes = tipo_minutos_restantes[:len(minutos_sin_tipo)]
                    random.shuffle(tipo_minutos_restantes)

            for minuto, tipo in zip(minutos_sin_tipo, tipo_minutos_restantes):
                rows.append({
                    'id_player_game': id_player_game,
                    'minuto': minuto,
                    'type': tipo
                })

            for minuto, tipo in minutos_con_tipo.items():
                rows.append({
                    'id_player_game': id_player_game,
                    'minuto': minuto,
                    'type': tipo
                })
            

        else:
            # Aquí procesamos cuando GM está vacío pero hay tipos
            for tipo in ['G', 'P', 'OG']:
                valor = row.get(tipo)
                if pd.notna(valor) and str(valor).strip() != '':
                    partes = str(valor).replace(',', ' ').split()
                    cantidad = len(partes)
                    for _ in range(cantidad):
                        print(f"ℹ️ Jugador {player_name}: tipo {tipo} sin minuto")
                        rows.append({
                            'id_player_game': id_player_game,
                            'minuto': 'N/A',
                            'type': tipo
                        })

        # Tarjetas sin minuto (igual que antes)
        for tarjeta_col, tarjeta_tipo in [('Tarjetas amarillas', 'A'), ('Tarjetas Rojas', 'R')]:
            tarjetas = row.get(tarjeta_col)
            if tarjetas in [None, '', np.nan]:
                tarjetas = 0
            try:
                cantidad = int(tarjetas)
            except (ValueError, TypeError):
                cantidad = 0

            if cantidad > 0:
                for _ in range(cantidad):
                    print(f"ℹ️ Jugador {player_name} recibió una tarjeta {'amarilla' if tarjeta_tipo == 'A' else 'roja'}")
                    rows.append({
                        'id_player_game': id_player_game,
                        'minuto': 'N/A',
                        'type': tarjeta_tipo
                    })

    df_Action_document_aux = pd.DataFrame(rows,columns=['id_player_game','minuto','type'])

    
    minutos_extraidos = df_Action_document_aux['minuto'].astype(str).str.extract(r'(\d+)')

    # Convertir a número, ignorando errores y NaNs (mantiene NaN en lugar de fallar)
    df_Action_document_aux['minuto_base'] = pd.to_numeric(minutos_extraidos[0], errors='coerce')

    # Asignar el half: 1 si minuto_base <= 45, 2 si 45 < minuto_base <= 90, NaN si fuera de rango
    df_Action_document_aux['half'] = np.select(
        [
            df_Action_document_aux['minuto_base'] <= 45,
            (df_Action_document_aux['minuto_base'] > 45) & (df_Action_document_aux['minuto_base'] <= 90)
        ],
        [1, 2],
        default=np.nan
    )
    df_Action_document_aux.rename(columns={'minuto_base': 'time'}, inplace=True)
    #df_Action_document_aux = df_Action_document_aux.drop(columns=['minuto'])
    df_Action_document_aux['id_action_document'] = [str(uuid.uuid4()) for _ in range(len(df_Action_document_aux))]


    df_Action_document=pd.concat([df_Action_document,df_Action_document_aux], ignore_index=True)
    df_Action_document.to_excel(r'..\Maestros_BD\Action_document.xlsx', index=False, sheet_name='Action_document')

In [6]:
def Team_game(id_match,name_match,result,id_team_home,id_team_away):

    id_team_match_home=uuid.uuid4()
    id_team_match_away=uuid.uuid4()

    teams=name_match.split(' - ')
    name_home_team=teams[0].strip()
    name_away_team=teams[1].strip()
    goals=result.split(' - ')
    home_goals=goals[0].strip()
    away_goals=goals[1].strip()

    if(home_goals>away_goals):
        home_result='W'
        away_result='L'
    elif(home_goals<away_goals):
        home_result='L'
        away_result='W'
    else:
        home_result='D'
        away_result='D'


    #Falta añadir si es posible las tarjetas
    
    df_Team_game.loc[len(df_Team_game)] = [id_match, id_team_home,id_team_match_home, name_home_team,home_result,home_goals,away_goals]
    df_Team_game.loc[len(df_Team_game)] = [id_match, id_team_away,id_team_match_away, name_away_team,away_result,away_goals,home_goals]

    df_Team_game.to_excel(r'..\Maestros_BD\Team_game.xlsx',index=False,sheet_name='Team_game')

    return id_team_match_home,id_team_match_away

In [7]:
def Stadium_team(data):
        global df_Stadium
        id_stadium=''
        stadium_name = data.get('Stadium:', '').strip()
        if not stadium_name:
            id_stadium=''
        
        elif (not df_Stadium['stadium_name'].isin([stadium_name]).any()):
            id_stadium=uuid.uuid4()
            df_Stadium_aux = pd.DataFrame([[
                id_stadium,
                stadium_name,
                data.get('Capacity:', ''),
                data.get('Stadium Address:', ''),
                data.get('url_map', ''),
                data.get('Type of Pitch:', '')
            ]],columns=['id_stadium', 'stadium_name','capacity_sucio','stadium_adress','url_map', 'pitch_type'])

            df_Stadium_aux['pitch_type'] = (
                df_Stadium_aux['pitch_type']
                .str.replace(' x ', 'x', regex=False)
                .str.replace(' * ', 'x', regex=False)
                .str.replace(' X ', 'x', regex=False)
            )
            df_split = df_Stadium_aux['pitch_type'].str.split(' ', expand=True)
            df_split.columns = ['type_of_pitch', 'dimensions']

            # Unirlo al DataFrame original
            df_Stadium_aux = pd.concat([df_Stadium_aux, df_split], axis=1)


            df_Stadium_aux['capacity_sucio'] = df_Stadium_aux['capacity_sucio'].str.replace(',', '', regex=False)

            df_split = df_Stadium_aux['capacity_sucio'].str.split('Covered Seats:', expand=True)
            df_split.columns = ['capacity', 'covered_seats']
            df_Stadium_aux = pd.concat([df_Stadium_aux, df_split], axis=1)

            df_Stadium_aux['dimensions'] = df_Stadium_aux['dimensions'].str.replace('m', '', regex=False)
            df_Stadium_aux.drop(columns='capacity_sucio', inplace=True)
            df_Stadium_aux.drop(columns='pitch_type', inplace=True)

            df_Stadium = pd.concat([df_Stadium, df_Stadium_aux], ignore_index=True)
            df_Stadium.to_excel(r'..\Maestros_BD\Stadium.xlsx',index=False,sheet_name='Stadium')

        else:
            id_stadium=df_Stadium[(df_Stadium['stadium_name']==stadium_name)]['id_stadium'].values[0]

In [8]:
def Stadium_match(stadium_name):
    global df_Stadium    
    if (not df_Stadium['stadium_name'].isin([stadium_name]).any()):
        id_stadium=uuid.uuid4()

        df_Stadium_aux=pd.DataFrame([[id_stadium,stadium_name,'','','','', '','']],columns=['id_stadium', 'stadium_name','capacity','covered_seats','stadium_adress','url_map', 'type_of_pitch','dimensions'])
        df_Stadium = pd.concat([df_Stadium, df_Stadium_aux], ignore_index=True)
        df_Stadium.to_excel(r'..\Maestros_BD\Stadium.xlsx',index=False,sheet_name='Stadium')
        
    else:
        id_stadium=df_Stadium[(df_Stadium['stadium_name']==stadium_name)]['id_stadium'].values[0]       

    return id_stadium

In [9]:
def Players(players_sucio):
    global url_general,df_Player
    header = {'User-Agent':str(ua.random)}


    for i in range(len(players_sucio)):
        name_player=players_sucio[i].text.strip()

        if (not df_Player['name_player'].isin([name_player]).any()):

            url_players= url_general + players_sucio[i].get('href')

            session_players = requests.Session()
            session_players.headers.update(header)
            resultado_inicio_players = session_players.get(url_players)
            contenido_players = resultado_inicio_players.content.decode('utf-8', errors='ignore')

            bs_inicio_players = BeautifulSoup(contenido_players, 'lxml')


            table=bs_inicio_players.find(class_='table table-striped')

            rows = table.find_all('tr')
            
            player_info = {}
            for row in rows:
                cols = row.find_all('td')
                if len(cols) >= 2:
                    key = cols[0].get_text(strip=True)
                    value = cols[1].get_text(strip=True)
                    player_info[key] = value
                    
            id_player=uuid.uuid4()

            df_Player_aux = pd.DataFrame([[
                id_player,
                name_player,
                player_info.get('Usual Position:', ''),
                player_info.get('Age:', ''),
                player_info.get('Place\xa0of Birth:', ''),
                player_info.get('Joined:', '')
            ]], columns=['id_player', 'name_player', 'position', 'age', 'place_of_birth', 'joined'])

            df_Player = pd.concat([df_Player, df_Player_aux], ignore_index=True)
            df_Player.to_excel(r'..\Maestros_BD\Player.xlsx',index=False,sheet_name='Player')
    
        else:
            id_player=df_Player[df_Player['name_player']==name_player]['id_player'].values[0]

             




In [10]:
def Staff(name_staff):
    global df_Staff
    if not df_Staff['name_staff'].isin([name_staff]).any():
        id_staff=uuid.uuid4()

        df_Staff_aux=pd.DataFrame([[id_staff,name_staff]],columns=['id_staff', 'name_staff'])

        df_Staff = pd.concat([df_Staff, df_Staff_aux], ignore_index=True)
        df_Staff.to_excel(r'..\Maestros_BD\Staff.xlsx',index=False,sheet_name='Staff')

        
    else:
        id_staff=df_Staff[df_Staff['name_staff']==name_staff]['id_staff'].values[0]

    return id_staff

In [11]:
def Staff_team(data,id_team):
    global df_Staff_team

    claves_deseadas = [
    'President:', 'Chairman:', 'Manager:', 'Football Secretary:',
    'Fixtures Secretary:', 'Press Officer:', 'Programme Editor:',
    'Club Safety Officer:', 'Therapist:'
    ]

    datos_filtrados = {k.rstrip(':'): data[k] for k in claves_deseadas if k in data}

    # Crear DataFrame con los datos nuevos
    df_Staff_team_aux = pd.DataFrame(list(datos_filtrados.items()), columns=['role', 'name_staff'])
    df_Staff_team_aux['id_team'] = id_team

    # Generar id_staff para cada fila usando apply
    df_Staff_team_aux['id_staff'] = df_Staff_team_aux['name_staff'].apply(lambda name: Staff(str(name)))

    # Eliminar columna auxiliar
    df_Staff_team_aux = df_Staff_team_aux.drop(columns=['name_staff'])

    # Eliminar duplicados respecto a df_Staff_team (si existe)
    if not df_Staff_team.empty:
        # Hacemos merge para encontrar filas que ya existen
        merged = df_Staff_team_aux.merge(df_Staff_team, on=['role', 'id_staff', 'id_team'], how='left', indicator=True)
        nuevas_filas = merged[merged['_merge'] == 'left_only'].drop(columns=['_merge'])

        if nuevas_filas.empty:
            print("Ya existe")
        else:
            df_Staff_team = pd.concat([df_Staff_team, nuevas_filas], ignore_index=True)
            print(f"{len(nuevas_filas)} fila(s) nueva(s) añadida(s)")
    else:
        # Si df_Staff_team está vacío, simplemente añadimos todo
        df_Staff_team = pd.concat([df_Staff_team, df_Staff_team_aux], ignore_index=True)
        print("Todas las filas añadidas (DataFrame estaba vacío)")

    df_Staff_team.to_excel(r'..\Maestros_BD\Staff_team.xlsx',index=False,sheet_name='Staff_team')        


In [12]:
def Match_players(id_team_home,id_team_away,tablas):
    global df_Player_game,df_Player

    if tablas:
        table_html = str(tablas[0])
        table_html = table_html.replace('<i class="fa fa-file-text yellow-card"></i>', '1')
        table_html = table_html.replace('<i class="fa fa-file-text red-card"></i>', '1')
        table_html = table_html.replace('<i class="fa fa-soccer-ball-o"></i>', '1')
        table_html = table_html.replace('<i class="fa fa-soccer-ball-o green-text"></i>', '1')
    
    if 'table_html' in locals() and table_html.strip():
        html_data = StringIO(table_html)

        df = pd.read_html(html_data)[0]
        
        indice_substitutes = df[df['Player'] == 'Substitutes'].index[0]

        df['Status'] = df.apply(lambda row: 'Titular' if row.name <= indice_substitutes else 'Suplente', axis=1)


        df_local = df.iloc[:, :8].join(df.iloc[:, [-1]])


        df_local.rename(columns={'Unnamed: 6': 'Tarjetas amarillas', 'Unnamed: 7': 'Tarjetas Rojas'}, inplace=True)

        df_visitante = df.iloc[:, 8:]
        df_visitante.rename(columns={'Unnamed: 14': 'Tarjetas amarillas', 'Unnamed: 15': 'Tarjetas Rojas'}, inplace=True)

        df_visitante.rename(columns={'Player.1': 'Player', 'G.1': 'G','GM.1': 'GM', 'P.1': 'P', 'OG.1': 'OG'}, inplace=True)

        df_visitante = df_visitante.iloc[:, :8].join(df.iloc[:, [-1]])


        df_local['id_team']=id_team_home
        df_visitante['id_team']=id_team_away

        df_local['minuto_inicio'] = df_local['Status'].apply(lambda x: 0 if x == 'Titular' else "")
        df_visitante['minuto_inicio'] = df_visitante['Status'].apply(lambda x: 0 if x == 'Titular' else "")

        try:
            df_substitutes_local = df_local[df_local['Player'].str.contains('Subst by:', na=False)].reset_index(drop=True)
            df_substitutes_local=df_substitutes_local[['Player']]
            df_substitutes_local[['Player_sale', 'Player_entra_aux']] = df_substitutes_local['Player'].str.split('Subst by: ', expand=True)
            df_substitutes_local[['Player_entra', 'Time']] = df_substitutes_local['Player_entra_aux'].str.extract(r'([A-Za-z\s\-]+)\s\((\d+\smins )\)')
            df_substitutes_local['Time'] = df_substitutes_local['Time'].str.replace(' mins', '', regex=False)
            df_substitutes_local.drop(['Player', 'Player_entra_aux'], axis=1, inplace=True)
            df_local[['Player_sale', 'Player_entra_aux']] = df_local['Player'].str.split('Subst by: ', expand=True)
            df_local.drop(['Player', 'Player_entra_aux'], axis=1, inplace=True)
            df_local.rename(columns={'Player_sale': 'Player'}, inplace=True)




            df_local['minuto_sale'] = df_local['Player'].apply(
                lambda x: df_substitutes_local[df_substitutes_local['Player_sale'] == x]['Time'].values[0]
                if any(df_substitutes_local['Player_sale'] == x) else np.nan
            )

            df_local['minuto_suplentes'] = df_local['Player'].apply(
                lambda x: df_substitutes_local[df_substitutes_local['Player_entra'] == x]['Time'].values[0]
                if any(df_substitutes_local['Player_entra'] == x) else np.nan
            )
        except:
            df_local['minuto_sale'] = np.nan
            df_local['minuto_suplentes'] = np.nan
        try:
            df_substitutes_visitante = df_visitante[df_visitante['Player'].str.contains('Subst by:', na=False)].reset_index(drop=True)
            df_substitutes_visitante=df_substitutes_visitante[['Player']]
            df_substitutes_visitante[['Player_sale', 'Player_entra_aux']] = df_substitutes_visitante['Player'].str.split('Subst by: ', expand=True)
            df_substitutes_visitante[['Player_entra', 'Time']] = df_substitutes_visitante['Player_entra_aux'].str.extract(r'([A-Za-z\s\-]+)\s\((\d+\smins)\)')
            df_substitutes_visitante['Time'] = df_substitutes_visitante['Time'].str.replace(' mins', '', regex=False)
            df_substitutes_visitante.drop(['Player', 'Player_entra_aux'], axis=1, inplace=True)
            df_visitante[['Player_sale', 'Player_entra_aux']] = df_visitante['Player'].str.split('Subst by: ', expand=True)
            df_visitante.drop(['Player', 'Player_entra_aux'], axis=1, inplace=True)
            df_visitante.rename(columns={'Player_sale': 'Player'}, inplace=True)

            df_visitante['minuto_sale'] = df_visitante['Player'].apply(
                lambda x: df_substitutes_visitante[df_substitutes_visitante['Player_sale'] == x]['Time'].values[0]
                if any(df_substitutes_visitante['Player_sale'] == x) else np.nan
            )
            df_visitante['minuto_suplentes'] = df_visitante['Player'].apply(
                lambda x: df_substitutes_visitante[df_substitutes_visitante['Player_entra'] == x]['Time'].values[0]
                if any(df_substitutes_visitante['Player_entra'] == x) else np.nan
            )

        except:
            df_visitante['minuto_sale'] = np.nan
            df_visitante['minuto_suplentes'] = np.nan

        
        df_visitante['minuto_suplentes']=df_visitante['minuto_suplentes'].fillna('')
        df_visitante['minuto_sale']=df_visitante['minuto_sale'].fillna('')
        df_visitante['Tarjetas amarillas']=df_visitante['Tarjetas amarillas'].fillna('')
        df_visitante['Tarjetas Rojas']=df_visitante['Tarjetas Rojas'].fillna('')
        df_visitante['NewColumn'] = df_visitante['minuto_inicio'].astype(str) + df_visitante['minuto_suplentes'].astype(str)


        df_local = df_local[(df_local['Player'] != 'Substitutes')&(df_local['Player'] != '')]

        df_visitante = df_visitante[(df_visitante['Player'] != 'Substitutes')&(df_visitante['Player'] != '')]





        df_local['minuto_suplentes']=df_local['minuto_suplentes'].fillna('')
        df_local['minuto_sale']=df_local['minuto_sale'].fillna('')
        df_local['Tarjetas amarillas']=df_local['Tarjetas amarillas'].fillna('')
        df_local['Tarjetas Rojas']=df_local['Tarjetas Rojas'].fillna('')
        df_local['NewColumn'] = df_local['minuto_inicio'].astype(str) + df_local['minuto_suplentes'].astype(str)

        df_local['minuto_suplentes']=df_local['minuto_suplentes'].fillna('')
        df_local['minuto_sale']=df_local['minuto_sale'].fillna('')
        df_local['Tarjetas amarillas']=df_local['Tarjetas amarillas'].fillna('')
        df_local['Tarjetas Rojas']=df_local['Tarjetas Rojas'].fillna('')
        df_local['NewColumn'] = df_local['minuto_inicio'].astype(str) + df_local['minuto_suplentes'].astype(str)





        df_local.drop(['minuto_inicio', 'minuto_suplentes','No.'], axis=1, inplace=True)
        df_local.rename(columns={'NewColumn': 'minuto_inicio'}, inplace=True)

        df_local['minuto_sale'] = df_local.apply(
            lambda row: 90 if ((row['minuto_sale'] == '' and row['Status'] == 'Titular' and row['Tarjetas Rojas'] == '') or 
                            (row['Status'] == 'Suplente' and row['minuto_inicio'] != '' and row['Tarjetas Rojas'] == '')) 
                    else row['minuto_sale'], 
            axis=1
        )

        df_visitante.drop(['minuto_inicio', 'minuto_suplentes','#'], axis=1, inplace=True)
        df_visitante.rename(columns={'NewColumn': 'minuto_inicio'}, inplace=True)

        df_visitante['minuto_sale'] = df_visitante.apply(
            lambda row: 90 if ((row['minuto_sale'] == '' and row['Status'] == 'Titular' and row['Tarjetas Rojas'] == '') or 
                            (row['Status'] == 'Suplente' and row['minuto_inicio'] != '' and row['Tarjetas Rojas'] == '')) 
                    else row['minuto_sale'], 
            axis=1
        )

        df_local['Goles'] = df_local['GM'].apply(lambda x: len(str(x).split(',')) if isinstance(x, str) else 0)
        df_visitante['Goles'] = df_visitante['GM'].apply(lambda x: len(str(x).split(',')) if isinstance(x, str) else 0)
        df_local['id_player_game'] = [uuid.uuid4() for _ in range(len(df_local))]

        df_visitante['id_player_game'] = [uuid.uuid4() for _ in range(len(df_visitante))]

        Action_document(df_local,df_visitante)

        df_local.drop(['G', 'GM', 'P'], axis=1, inplace=True)
        df_visitante.drop(['G', 'GM', 'P'], axis=1, inplace=True)

        df_local['Goles'] = pd.to_numeric(df_local['Goles'], errors='coerce')
        df_local['OG'] = pd.to_numeric(df_local['OG'], errors='coerce')

        df_local['Goles'] = df_local.apply(
            lambda row: row['Goles'] - row['OG'] if row['Goles'] > 0 and pd.notna(row['OG']) and row['OG'] != '' else row['Goles'],
            axis=1
        )

        df_visitante['Goles'] = pd.to_numeric(df_visitante['Goles'], errors='coerce')
        df_visitante['OG'] = pd.to_numeric(df_visitante['OG'], errors='coerce')

        df_visitante['Goles'] = df_visitante.apply(
            lambda row: row['Goles'] - row['OG'] if row['Goles'] > 0 and pd.notna(row['OG']) and row['OG'] != '' else row['Goles'],
            axis=1
        )

        df_local['OG']=df_local['OG'].fillna('')
        df_visitante['OG']=df_visitante['OG'].fillna('')

        df_local.rename(columns={'OG': 'OwnGoals'}, inplace=True)
        df_visitante.rename(columns={'OG': 'OwnGoals'}, inplace=True)

        df_local = df_local[['Player', 'id_team', 'Status', 'minuto_inicio', 'minuto_sale', 'Goles', 'OwnGoals', 'Tarjetas amarillas', 'Tarjetas Rojas']]  # Ajusta el orden según sea necesario
        df_visitante = df_visitante[['Player', 'id_team', 'Status', 'minuto_inicio', 'minuto_sale', 'Goles', 'OwnGoals', 'Tarjetas amarillas', 'Tarjetas Rojas']]  # Ajusta el orden según sea necesario

        df_combined = pd.concat([df_local, df_visitante], ignore_index=True)

        valor=1
    else:
        df_combined=pd.DataFrame()
        valor=0

    
    if valor==0:
            print('Partido aplazado')
    else:
        df_combined['minuto_inicio'] = pd.to_numeric(df_combined['minuto_inicio'], errors='coerce')
        df_combined['minuto_sale'] = pd.to_numeric(df_combined['minuto_sale'], errors='coerce')

        df_combined.loc[df_combined['minuto_inicio'] == 0, 'start_reason'] = 'Starting XI'
        df_combined.loc[(df_combined['minuto_inicio'] != 0)&(~ df_combined['minuto_inicio'].isna()), 'start_reason'] = 'Substitution - On'
        df_combined.loc[(df_combined['minuto_inicio'].isna())&(df_combined['minuto_sale'].isna()), 'start_reason'] = 'No Play'

        df_combined.loc[(df_combined['minuto_sale'] == 90)&(df_combined['Tarjetas Rojas'] !=1), 'end_reason'] = 'Final Whistle'
        df_combined.loc[(df_combined['minuto_sale'] != 90)&(df_combined['Tarjetas Rojas'] !=1)&(~ df_combined['minuto_inicio'].isna()), 'end_reason'] = 'Substitution - Off'
        df_combined.loc[df_combined['Tarjetas Rojas'] ==1, 'end_reason'] = 'Ejection'
        df_combined.loc[(df_combined['minuto_inicio'].isna())&(df_combined['minuto_sale'].isna()), 'end_reason'] = 'No Play'


        for i in range(len(df_combined)):
            name_player=str(df_combined['Player'][i])
            if not df_Player['name_player'].isin([name_player]).any():
                id_player=uuid.uuid4()
                
                df_Player_aux = pd.DataFrame([[
                    id_player,
                    name_player,
                    '',
                    '',
                    '',
                    ''
                ]], columns=['id_player', 'name_player', 'position', 'age', 'place_of_birth', 'joined'])

                df_Player = pd.concat([df_Player, df_Player_aux], ignore_index=True)
                df_Player.to_excel(r'..\Maestros_BD\Player.xlsx',index=False,sheet_name='Player')
            else:
                id_player=df_Player[df_Player['name_player']==name_player]['id_player'].values[0]

            df_combined.loc[i, 'id_player'] = id_player


        df_combined.drop(['Player'], axis=1, inplace=True)
 
        df_Player_game=pd.concat([df_Player_game,df_combined], ignore_index=True)
        df_Player_game.to_excel(r'..\Maestros_BD\Player_game.xlsx', index=False, sheet_name='Player_game')

    

In [13]:
def Teams(teams_url_sucio):
    global df_Team



    header = {'User-Agent':str(ua.random)}
    num_equipos=0
    for i in range(len(teams_url_sucio)):
        url_team=teams_url_sucio[i].find('a',href=True).get('href')
        url_team_split=url_team.split('/')
        id_team_web=str(url_team_split[3])

        url_team= url_general + url_team

        session_team = requests.Session()
        session_team.headers.update(header)
        resultado_inicio_team = session_team.get(url_team)
        contenido_team = resultado_inicio_team.content.decode('utf-8', errors='ignore')

        bs_inicio_team = BeautifulSoup(contenido_team, 'lxml')

        name_team=bs_inicio_team.find(class_='col-md-10 col-sm-9').text.strip()

        url_logo=url_general + bs_inicio_team.find(class_='col-md-2 col-sm-3 badge').find('img')['src']

        container=bs_inicio_team.find_all(class_='col-lg-6 col-md-6 col-sm-12 col-xs-12')

        url_map=bs_inicio_team.find_all(id='Squad',href=True)
        if(len(url_map)>1):
            url_map=url_map[1].get('href')
        else:
            url_map=url_map[0].get('href')

        url_web=bs_inicio_team.find(id='URL',href=True).get('href')

        data = {}
        for j in range(len(container)):
            for dl in container[j].find_all('dl', class_='dl-horizontal team'):
                dt = dl.find('dt').get_text(strip=True)
                dd = dl.find('dd').get_text(separator=",", strip=True)  # Usa salto de línea donde hay <br/>
                data[dt] = dd
        
        data['url_map'] = url_map

        Stadium_team(data)

        if not df_Team['name_team'].isin([name_team]).any():
            id_team=uuid.uuid4()
            nickname = data.get('Nickname:', '').strip()
            if nickname:
                df_Team_aux = pd.DataFrame([[
                    id_team,
                    id_team_web,
                    name_team,
                    nickname,
                    data.get('Founded:', ''),
                    data.get('Home Kit:', ''),
                    data.get('Goalkeeper Kit:', ''),
                    data.get('Away Kit:', ''),
                    data.get('Alternate Colours:', ''),
                    data.get('Tel:', ''),
                    data.get('Email:', ''),
                    url_logo,
                    url_web,
                    data.get('Club Sponsor:', '')
                ]],columns=['id_team','id_team_web', 'name_team','nickname','founded','home_kit', 'goalkeeper_kit','away_kit','alternate_colours','phone','email','url_logo','url_web','club_sponsor'])

                df_Team = pd.concat([df_Team, df_Team_aux], ignore_index=True)
                df_Team.to_excel(r'..\Maestros_BD\Team.xlsx',index=False,sheet_name='Team')

        
        else:
            id_team=df_Team[df_Team['name_team']==name_team]['id_team'].values[0]

        Staff_team(data,id_team)

        num_equipos=num_equipos+1

        if(num_equipos==1):
            id_team_home=id_team
        elif(num_equipos==2):
            id_team_away=id_team
        else:
            id_team_home=''
            id_team_away=''


    return id_team_home,id_team_away




In [14]:
def Match(match_basic_data_sucio,id_team_home,id_team_away,id_fase,id_match_web):
        
        id_match=uuid.uuid4()
        round=''
        url_video=''
        id_stadium=''

        name_match=match_basic_data_sucio.find(class_='margin-top-20').text.replace(' v ', ' - ').strip()
        result=match_basic_data_sucio.find(class_='score-xs hidden-lg hidden-md hidden-sm').text.replace('0 win  0 - 0 on penalties', '').strip()

        kick_off = attendance = name_referee = fixture_type = None
        start_time = start_date = estadio = id_stadium = None

        for p in match_basic_data_sucio.find_all('p'):
            for strong in p.find_all('strong'):
                label = strong.get_text(strip=True).lower().rstrip(':')
                content = strong.next_sibling
                if content:
                    content = content.strip()
                else:
                    content = ''

                if label == 'fixture type':
                    fixture_type = content
                    
                elif label == 'kick off':
                    kick_off = content
                    match = re.search(r'(\d{2}:\d{2}),\s*(.+?) at (.+)', kick_off)
                    if match:
                        start_time = match.group(1)
                        start_date = match.group(2)
                        start_date = start_date.split(',')[-1].strip()
                        estadio = match.group(3)
                        # Convierte estadio a id_stadium con tu función
                        id_stadium = Stadium_match(estadio)
                    else:
                        print("No se pudo hacer regex match con kick_off:", kick_off)
                elif label == 'attendance':
                    attendance = content
                elif label == 'referee':
                    name_referee = content
                    Referee(id_match, name_referee)

        df_Match.loc[len(df_Match)] = [id_match,id_match_web, id_fase, round, url_video, start_date, start_time, id_stadium, attendance, name_match, result]
        df_Match.to_excel(r'..\Maestros_BD\Match.xlsx', index=False, sheet_name='Match')

        id_team_match_home,id_team_match_away=Team_game(id_match,name_match,result,id_team_home,id_team_away)

        return id_match,id_team_match_home,id_team_match_away


In [15]:
#lista_meses=['Aug','Sep','Oct','Nov','Dec','Jan','Feb','Mar','Apr']
lista_meses=['Aug']
header = {'User-Agent':str(ua.random)}
df_final=pd.DataFrame()


for i in range(len(lista_meses)):

    #Entrar en la web y sacar todo el contenido
    print(lista_meses[i])
    website_inicio:str='https://southern-football-league.co.uk/Results/All/All/2025/2026/P/Southern%20League%20Premier%20Central/'+lista_meses[i]

    session = requests.Session()
    session.headers.update(header)
    resultado_inicio = session.get(website_inicio)
    contenido = resultado_inicio.content.decode('utf-8', errors='ignore')

    bs_inicio = BeautifulSoup(contenido, 'lxml')

    #Sacar la tabla de todos los partidos de esa url
    div_partidos=bs_inicio.find_all(class_='FixtureTypeMobile')


    for j in range(len(div_partidos)):
        #Sacar la competicion de cada partido de esa url
        competition= div_partidos[j].find('h5').text.strip()
        id_fase=Competition(bs_inicio, competition)


        #Entrar en la web de cada partido
        href = div_partidos[j].find_all('a',href=True)

        url_fin=href[0].get('href')
        url_split=url_fin.split('/')
        id_match_web=str(url_split[3])
        url_match=url_general+'/match/m/'+id_match_web+'/squad/'
        
        print(url_match)

        session_match = requests.Session()
        session_match.headers.update(header)
        resultado_match = session_match.get(url_match)
        contenido_match = resultado_match.content.decode('utf-8', errors='ignore')

        bs_inicio_match = BeautifulSoup(contenido_match, 'lxml')


        teams_url_sucio=bs_inicio_match.find_all(class_='col-md-2 col-sm-5 col-xs-5 badge')

        id_team_home,id_team_away=Teams(teams_url_sucio)

        #Sacar datos basicos del partido
        match_basic_data_sucio=bs_inicio_match.find(class_='col-md-6 col-sm-12 col-xs-12 fixture-info no-pad-xs')
        id_match,id_team_match_home,id_team_match_away=Match(match_basic_data_sucio,id_team_home,id_team_away,id_fase,id_match_web)
        
        tablas = bs_inicio_match.find_all(class_='match-report')

        players_sucio=bs_inicio_match.find_all('a', attrs={'target': '_top'},href=True)
        
        Players(players_sucio)
        
        Match_players(id_team_match_home,id_team_match_away,tablas)







Aug
Nueva competición detectada. Por favor introduce los datos:
Competicion: Southern League Prem Central
Nivel: 7
Tipo: League
Categoria: Senior
https://southern-football-league.co.uk//match/m/145131/squad/
Todas las filas añadidas (DataFrame estaba vacío)
Todas las filas añadidas (DataFrame estaba vacío)
Partido aplazado
https://southern-football-league.co.uk//match/m/145132/squad/
Ya existe
8 fila(s) nueva(s) añadida(s)
Partido aplazado
https://southern-football-league.co.uk//match/m/145133/squad/
Ya existe
8 fila(s) nueva(s) añadida(s)
Partido aplazado
https://southern-football-league.co.uk//match/m/145134/squad/
Ya existe
8 fila(s) nueva(s) añadida(s)
Partido aplazado
https://southern-football-league.co.uk//match/m/145135/squad/
Ya existe
8 fila(s) nueva(s) añadida(s)
Partido aplazado
https://southern-football-league.co.uk//match/m/145136/squad/
Ya existe
8 fila(s) nueva(s) añadida(s)
Partido aplazado
https://southern-football-league.co.uk//match/m/145137/squad/
Ya existe
8 fila(s